In [ ]:
import warnings; warnings.filterwarnings("ignore")
import torch, gc, os, json, re, time, random, numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer

random.seed(42); torch.manual_seed(42)
print(f"CUDA: {torch.version.cuda} | GPU: {torch.cuda.get_device_name(0)}")
os.environ["HF_TOKEN"] = "REDACTED"
print("HF token set")
# Verify CUDA actually works on this GPU
try:
    x = torch.zeros(1).cuda()
    print(f"CUDA OK: {x.device}")
except Exception as e:
    print(f"CUDA FAIL: {e}")


In [ ]:
ITEMS = [{"instr":"Explain how photosynthesis works.","resp":"Plants use sunlight to convert CO2 and water into glucose and oxygen."},{"instr":"What is the theory of relativity?","resp":"Einstein theory says space and time are relative."},{"instr":"Describe the water cycle.","resp":"Water evaporates, forms clouds, returns as precipitation."},{"instr":"What causes earthquakes?","resp":"Tectonic plates shift, releasing seismic waves."},{"instr":"Explain how vaccines work.","resp":"Vaccines train immune system to recognize pathogens."},{"instr":"What is DNA?","resp":"DNA carries genetic instructions for growth and reproduction."},{"instr":"Describe the solar system.","resp":"8 planets orbiting the Sun."},{"instr":"What is entropy?","resp":"Disorder measure. Always increases per 2nd law."},{"instr":"How do batteries work?","resp":"Chemical reactions create electron flow via electrolyte."},{"instr":"What is a black hole?","resp":"Gravity so strong light cannot escape."},{"instr":"What is machine learning?","resp":"AI learning patterns from data without explicitly programming."},{"instr":"Describe cloud computing.","resp":"On-demand computing resources over the internet."},{"instr":"What is an API?","resp":"Interface for different software to communicate with each other."},{"instr":"Explain encryption.","resp":"Algorithms that encode data using cryptographic keys."},{"instr":"What is a database index?","resp":"Structure speeding up data retrieval like a book index."},{"instr":"What is Python?","resp":"High-level readable interpreted programming language."},{"instr":"Explain the internet.","resp":"Global network of computers communicating via TCP/IP."},{"instr":"What is blockchain?","resp":"Distributed ledger recording transactions in immutable blocks."},{"instr":"What is an operating system?","resp":"Software managing hardware resources for applications."},{"instr":"Explain neural networks.","resp":"Computational systems inspired by biological neural networks."},{"instr":"What caused World War I?","resp":"Assassination of Archduke Franz Ferdinand triggered alliance system."},{"instr":"Explain democracy.","resp":"System where citizens vote for representatives."},{"instr":"What was the Renaissance?","resp":"14th-17th century cultural rebirth in Europe."},{"instr":"Describe capitalism.","resp":"Economic system with private ownership and profit motive."},{"instr":"What is the United Nations?","resp":"International organization for peace and cooperation."},{"instr":"Explain the Cold War.","resp":"Geopolitical tension between US and USSR 1947-1991."},{"instr":"What is ethics?","resp":"Branch of philosophy studying moral right and wrong."},{"instr":"Describe the Enlightenment.","resp":"18th century intellectual movement emphasizing reason."},{"instr":"What is the Constitution?","resp":"Supreme law establishing government structure and rights."},{"instr":"Explain globalization.","resp":"Increasing interconnectedness of world economies and cultures."},{"instr":"How to make a budget?","resp":"Track income and expenses, allocate for needs and savings."},{"instr":"What is healthy eating?","resp":"Balanced diet with fruits, vegetables, protein, whole grains."},{"instr":"Explain first aid for burns.","resp":"Cool with running water, cover with sterile dressing."},{"instr":"What is recycling?","resp":"Converting waste materials into new reusable products."},{"instr":"How does a car engine work?","resp":"Fuel combustion in cylinders drives pistons turning crankshaft."},{"instr":"What is renewable energy?","resp":"Energy from replenishable sources like sun and wind."},{"instr":"How to grow vegetables?","resp":"Plant seeds in soil with water, sunlight, and nutrients."},{"instr":"What is a mortgage?","resp":"Loan for purchasing property paid in monthly installments."},{"instr":"Explain the stock market.","resp":"Platform where company shares are bought and sold."},{"instr":"What is inflation?","resp":"General increase in prices reducing purchasing power."},{"instr":"What is the Pythagorean theorem?","resp":"a squared plus b squared equals c squared."},{"instr":"Explain a prime number.","resp":"Number divisible only by 1 and itself."},{"instr":"What is calculus?","resp":"Mathematics of change using derivatives and integrals."},{"instr":"What is a logarithm?","resp":"Inverse of exponentiation, log base 10 of 100 is 2."},{"instr":"What is the quadratic formula?","resp":"x equals negative b plus or minus square root of b squared minus 4ac over 2a."},{"instr":"Explain probability.","resp":"Likelihood of events occurring, from 0 to 1."},{"instr":"What is a proof?","resp":"Logical demonstration that a statement follows from axioms."},{"instr":"What is a function?","resp":"Relation where each input maps to exactly one output."},{"instr":"Explain standard deviation.","resp":"Measure of dispersion from the mean."},{"instr":"What is a matrix?","resp":"Rectangular array of numbers for linear transformations."},]
print(f"{len(ITEMS)} items")


In [ ]:
PROBE_VARIANTS = {"rubric_order": ["control", "reversed", "random"], "score_id": ["numeric", "letter", "descriptive"], "reference_answer": ["none", "good", "poor"]}

def make_prompt(item, probe_type, variant):
    i, r = item["instr"], item["resp"]
    if probe_type == "rubric_order":
        rubric = {"control":"Score from 1-5 (where 1 is worst, 5 is best)", "reversed":"Score from 1-5 (where 1 is best, 5 is worst)", "random":"Score from 1-5"}[variant]
        return f"Evaluate the following response.\n### Instruction: {i}\n### Response: {r}\n### {rubric}\n### Score:"
    elif probe_type == "score_id":
        rubric = {"numeric":"Score from 1-5", "letter":"Score A-E (where A is best, E is worst)", "descriptive":"Rate: Poor, Fair, Good, Very Good, Excellent"}[variant]
        return f"Evaluate the following response.\n### Instruction: {i}\n### Response: {r}\n### {rubric}\n### Score:"
    elif probe_type == "reference_answer":
        ref = "" if variant == "none" else "\n### Excellent response: This is a perfect answer." if variant == "good" else "\n### Poor response: This is wrong."
        return f"Evaluate the following response.\n### Instruction: {i}{ref}\n### Response: {r}\n### Score from 1-5 (where 1 is worst, 5 is best)\n### Score:"
print("Probes ready")


In [ ]:
LETTER_MAP = {"A":5,"B":4,"C":3,"D":2,"E":1,"F":1}
DESCRIPTIVE_MAP = {"excellent":5,"very good":4,"good":3,"fair":2,"poor":1,"great":5,"average":3,"bad":1,"terrible":1,"perfect":5,"outstanding":5,"exceptional":5,"satisfactory":3,"inadequate":1,"mediocre":2}

def parse_score(text, variant):
    t = text.strip().lower()
    numbers = re.findall(r"\b([1-5])\b", t)
    letter_match = re.search(r"\b([a-f])\b", t)
    desc_score = None
    for word, score in sorted(DESCRIPTIVE_MAP.items(), key=lambda x:-len(x[0])):
        if word in t: desc_score = float(score); break
    
    if variant == "numeric":
        if numbers: return float(numbers[0])
        if letter_match: return float(LETTER_MAP.get(letter_match.group(1).upper(), 0))
        return desc_score
    elif variant == "letter":
        if letter_match: return float(LETTER_MAP.get(letter_match.group(1).upper(), 0))
        if numbers: return float(numbers[0])
        return desc_score
    elif variant == "descriptive":
        if desc_score: return desc_score
        if numbers: return float(numbers[0])
        if letter_match: return float(LETTER_MAP.get(letter_match.group(1).upper(), 0))
        return None
    else:
        if numbers: return float(numbers[0])
        if letter_match: return float(LETTER_MAP.get(letter_match.group(1).upper(), 0))
        return desc_score

def score_model(model, tokenizer, prompt, probe_variant, device):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=15, temperature=0.0, do_sample=False, pad_token_id=tokenizer.eos_token_id, num_return_sequences=1)
    full = tokenizer.decode(outputs[0], skip_special_tokens=True)
    generated = full[len(prompt):].strip()
    
    if "### Score:" in generated:
        after_score = generated.split("### Score:")[-1].split("###")[0].strip()
        s = parse_score(after_score, probe_variant)
        if s is not None and 1 <= s <= 5: return s
    s = parse_score(generated, probe_variant)
    if s is not None and 1 <= s <= 5: return s
    full_numbers = re.findall(r"### Score:?\s*([1-5])", full)
    if full_numbers: return float(full_numbers[-1])
    return None

print("Parser ready")


In [ ]:
print("fp16 + CPU offload (no bitsandbytes)")


In [ ]:
MODELS_7B = [
    ("mistralai/Mistral-7B-v0.3", "mistralai/Mistral-7B-Instruct-v0.3", "Mistral-7B-v0.3"),
    ("deepseek-ai/deepseek-llm-7b-base", "deepseek-ai/deepseek-llm-7b-chat", "DeepSeek-7B"),
]
MODELS_SMALL = [
    ("Qwen/Qwen2.5-0.5B", "Qwen/Qwen2.5-0.5B-Instruct", "Qwen2.5-0.5B"),
    ("Qwen/Qwen2.5-3B", "Qwen/Qwen2.5-3B-Instruct", "Qwen2.5-3B"),
]
DEEPSEEK_IDS = {"deepseek-ai/deepseek-llm-7b-chat"}
print(f"7B: {len(MODELS_7B)}  Small: {len(MODELS_SMALL)}")


In [ ]:
all_results = {}
device = torch.device("cuda")

for families, is_7b in [(MODELS_7B, True), (MODELS_SMALL, False)]:
    for base_id, instruct_id, name in families:
        ids_to_run = [(base_id, name)]
        if instruct_id:
            ids_to_run.append((instruct_id, f"{name}-IT"))
        
        for model_id, model_name in ids_to_run:
            print(f"\n{'='*60}\n{model_name}  ({model_id})\n{'='*60}")
            try:
                if is_7b:
                    load_kwargs = {"torch_dtype": torch.float16, "device_map": "auto", "max_memory": {0: "14.5GiB", "cpu": "32GiB"}, "offload_folder": "offload_temp"}
                    print("  mode: fp16 + CPU offload")
                else:
                    load_kwargs = {"torch_dtype": torch.float16, "device_map": "auto"}
                    print("  mode: fp16")

                tokenizer = AutoTokenizer.from_pretrained(model_id, token=os.environ["HF_TOKEN"])
                if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
                model = AutoModelForCausalLM.from_pretrained(model_id, **load_kwargs, token=os.environ["HF_TOKEN"])
                model.eval()
                print(f"  OK: {sum(p.numel() for p in model.parameters())/1e6:.0f}M parameters")
            except Exception as e:
                print(f"  FAILED: {e}")
                continue

            use_ct = model_id in DEEPSEEK_IDS
            scores = {pt: {} for pt in PROBE_VARIANTS}
            for pt, variants in PROBE_VARIANTS.items():
                for pv in variants:
                    item_scores = []
                    for idx, item in enumerate(ITEMS):
                        prompt = make_prompt(item, pt, pv)
                        if use_ct:
                            messages = [{"role": "user", "content": prompt}]
                            prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
                        reps = []
                        for _ in range(3):
                            try:
                                s = score_model(model, tokenizer, prompt, pv, device)
                                if s is not None and 1 <= s <= 5: reps.append(s)
                            except: pass
                        if reps: item_scores.append(np.mean(reps))
                        if (idx+1) % 25 == 0:
                            print(f"  [{pt}/{pv}] {idx+1}/{len(ITEMS)}  ({len(item_scores)} scored)", flush=True)
                    scores[pt][pv] = {"mean": float(np.mean(item_scores)) if item_scores else None}
            result = {}
            for pt, pv in PROBE_VARIANTS.items():
                means = [scores[pt][v]["mean"] for v in pv if scores[pt][v]["mean"] is not None]
                result[pt] = round(max(means)-min(means), 2) if len(means) >= 2 else None
            all_results[model_name] = result
            print(f"  DONE: {result}")
            del model; gc.collect(); torch.cuda.empty_cache(); time.sleep(3)

with open("final_results.json", "w") as f: json.dump(all_results, f, indent=2)
print(f"\nDONE: {len(all_results)} models")


In [ ]:
with open("final_results.json") as f: data = json.load(f)
print(f"=== {len(data)} MODELS ===")
for name, r in sorted(data.items()):
    bad = [f"{k}={v}" for k,v in r.items() if v is None]
    clean = [f"{k}={v}" for k,v in r.items() if v is not None]
    print(f"  {"\U000026a0" if bad else "\u2705"} {name:30s}  {"  ".join(clean)}")
import shutil; shutil.copy("final_results.json", "/kaggle/working/final_results.json")
